In [ ]:
import requests
from typing import List, Dict


class WikipediaTools:
    """
    Tools for interacting with the Wikipedia API.
    """

def search_wikipedia(self, query: str) -> List[Dict]:
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query
    }

    headers = {
        "User-Agent": "MyWikipediaAgent/1.0 (nirajan@example.com)"
    }

    response = requests.get(url, params=params, headers=headers)

    # Check status
    if response.status_code != 200:
        raise Exception(f"HTTP Error: {response.status_code}")

    # Ensure valid JSON
    try:
        data = response.json()
    except Exception:
        print("Response text was:", response.text[:500])
        raise Exception("Invalid JSON response from Wikipedia")

    return data["query"]["search"]

def get_page(self, title: str) -> str:
    url = "https://en.wikipedia.org/w/index.php"
    params = {
        "title": title,
        "action": "raw"
    }

    headers = {
        "User-Agent": "MyWikipediaAgent/1.0 (nirajan@example.com)"
    }

    response = requests.get(url, params=params, headers=headers)

    if response.status_code != 200:
        raise Exception(f"HTTP Error: {response.status_code}")

    return response.text

In [15]:
search_tool_schema = {
    "type": "function",
    "name": "search_wikipedia",
    "description": "Search Wikipedia for relevant pages.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search term"
            }
        },
        "required": ["query"]
    }
}

get_page_tool_schema = {
    "type": "function",
    "name": "get_page",
    "description": "Fetch raw content of a Wikipedia page.",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "Exact page title"
            }
        },
        "required": ["title"]
    }
}

In [16]:
import json
from openai import OpenAI


class Agent:

    def __init__(self, model, instructions, tools, tool_schemas):
        self.client = OpenAI()
        self.model = model
        self.instructions = instructions
        self.tools = tools
        self.tool_schemas = tool_schemas

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == "search_wikipedia":
            result = self.tools.search_wikipedia(**arguments)
        elif name == "get_page":
            result = self.tools.get_page(**arguments)
        else:
            result = f'Unknown tool {name}'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }

    def loop(self, user_prompt, message_history=None):

        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions}
            ]

        message_history.append({"role": "user", "content": user_prompt})

        while True:

            response = self.client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tool_schemas
            )

            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:

                if message.type == "function_call":
                    tool_output = self.make_call(message)
                    message_history.append(tool_output)
                    has_function_calls = True

                if message.type == "message":
                    print("ASSISTANT:", message.content[0].text)

            if not has_function_calls:
                break

        return message_history

In [17]:
instructions = """
You are a Wikipedia research assistant.

Follow 3 iterations:

1) First iteration:
   - Perform one search.
   - Explain why this query is appropriate.

2) Second iteration:
   - Analyze results.
   - Fetch the most relevant page.

3) Third iteration:
   - Synthesize a final answer.

Use only Wikipedia tool results.
If information is not found, say so clearly.
"""

In [18]:
wiki_tools = WikipediaTools()

agent = Agent(
    model="gpt-4o-mini",
    instructions=instructions,
    tools=wiki_tools,
    tool_schemas=[search_tool_schema, get_page_tool_schema]
)

In [21]:
results = wiki_tools.search_wikipedia("capybara")

print("Total results returned:", len(results))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)